# Laboratorio 05 - Naive Bayes para regresion de precio
## CC3074 - Mineria de Datos | Universidad del Valle de Guatemala

Dataset: `data/listings.RData`

---
## Actividad 1 - Modelo de regresion Naive Bayes para predecir el precio

En scikit-learn no existe un `NaiveBayesRegressor` nativo. Para resolverlo de forma metodologica, se usa un enfoque de dos pasos:
1) discretizar `price_num` en bins (clases) en entrenamiento,
2) entrenar `GaussianNB` para predecir el bin y luego convertir ese bin a un precio numerico usando la mediana del bin.

Para mantener comparabilidad con `../MD-LAB04/main.ipynb`, se replica el mismo preprocesamiento y la misma logica de division (`test_size=0.2`, `random_state=42`, `shuffle=True`, estratificado por deciles de `price_num`).

In [26]:
# Importaciones base
from pathlib import Path
import numpy as np
import pandas as pd
import pyreadr

# Cargar dataset desde la ruta del laboratorio
rdata_path = Path('data/listings.RData')
result = pyreadr.read_r(str(rdata_path))
df = result['listings'].copy()

print('Objeto cargado:', list(result.keys()))
print('Dimensiones originales:', df.shape)

Objeto cargado: ['listings']
Dimensiones originales: (171748, 80)


### Preprocesamiento y splits (replica de LAB04)

Se mantiene la misma seleccion de variables candidatas, limpieza de `price_num`, recorte al percentil 99, one-hot encoding y division train/test estratificada por deciles. Esto conserva la comparabilidad de resultados entre laboratorios.

In [27]:
# Utilidad para convertir montos de texto a numerico
def parse_money(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.replace(r'[$,]', '', regex=True).str.strip(),
        errors='coerce',
    )

# Copia de trabajo para EDA/modelado
df_eda = df.copy()

# Convertir precio a numerico
if 'price' in df_eda.columns:
    df_eda['price_num'] = parse_money(df_eda['price'])

# Convertir porcentajes host a numerico
for col in ['host_response_rate', 'host_acceptance_rate']:
    if col in df_eda.columns:
        df_eda[f'{col}_num'] = pd.to_numeric(
            df_eda[col].astype(str).str.replace('%', '', regex=False),
            errors='coerce',
        )

# Variables candidatas (mismas de LAB04)
candidate_features = [
    'room_type', 'property_type', 'neighbourhood_cleansed',
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'minimum_nights', 'maximum_nights', 'availability_365',
    'number_of_reviews', 'reviews_per_month', 'review_scores_rating',
    'host_is_superhost', 'host_identity_verified',
    'host_response_rate_num', 'host_acceptance_rate_num',
    'calculated_host_listings_count', 'instant_bookable',
    'latitude', 'longitude', 'price_num',
]

selected_cols = [c for c in candidate_features if c in df_eda.columns]
model_df = df_eda[selected_cols].copy()

# Limpieza del objetivo y recorte de extremos (mismo criterio LAB04)
model_df = model_df[model_df['price_num'].notna()].copy()
p99_price = model_df['price_num'].quantile(0.99)
model_df = model_df[model_df['price_num'] <= p99_price].copy()

# Imputacion de numericas y normalizacion de categoricas
num_cols = [c for c in model_df.select_dtypes(include=np.number).columns if c != 'price_num']
cat_cols = model_df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

for col in num_cols:
    model_df[col] = model_df[col].fillna(model_df[col].median())

for col in cat_cols:
    model_df[col] = model_df[col].astype(str).replace('nan', 'SinDato')

# Matriz de entrada final
X = model_df.drop(columns=['price_num'])
y = model_df['price_num']
X_encoded = pd.get_dummies(X, drop_first=True)

# Split identico a LAB04
from sklearn.model_selection import train_test_split

y_bins_split = pd.qcut(y, q=10, duplicates='drop')
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y_bins_split,
)

print('Shape X total:', X_encoded.shape)
print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('y_train:', y_train.shape, '| y_test:', y_test.shape)
print('Primeros 5 indices train:', X_train.index[:5].tolist())
print('Primeros 5 indices test:', X_test.index[:5].tolist())

Shape X total: (75531, 520)
X_train: (60424, 520) | X_test: (15107, 520)
y_train: (60424,) | y_test: (15107,)
Primeros 5 indices train: [23421, 36797, 25953, 160615, 27075]
Primeros 5 indices test: [41251, 54191, 163573, 29592, 23803]


### Entrenamiento del modelo Naive Bayes (enfoque para regresion)

Se discretiza `y_train` en 10 bins por cuantiles. Luego `GaussianNB` predice el bin para cada fila de test. Finalmente, cada bin predicho se transforma a valor continuo usando la mediana observada del bin en entrenamiento.

In [28]:
# Importar modelo y metricas
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1) Discretizar objetivo de entrenamiento en 10 bins por cuantiles
y_bins_train = pd.qcut(y_train, q=10, duplicates='drop')
y_bins_train_str = y_bins_train.astype(str)

# 2) Crear mapeo bin -> mediana de precio (usado para volver a escala continua)
bin_to_median = y_train.groupby(y_bins_train_str).median().to_dict()

# 3) Entrenar GaussianNB
nb_reg_proxy = GaussianNB()
nb_reg_proxy.fit(X_train, y_bins_train_str)

# 4) Predecir bins en test y convertirlos a precio numerico
y_bins_pred = pd.Series(nb_reg_proxy.predict(X_test), index=X_test.index)
y_pred_nb = y_bins_pred.map(bin_to_median).astype(float)

# 5) Baseline sencillo para contexto (siempre predecir mediana de train)
y_pred_baseline = pd.Series(y_train.median(), index=y_test.index)

# 6) Calcular metricas de regresion
mae_nb = mean_absolute_error(y_test, y_pred_nb)
rmse_nb = np.sqrt(mean_squared_error(y_test, y_pred_nb))
r2_nb = r2_score(y_test, y_pred_nb)

mae_base = mean_absolute_error(y_test, y_pred_baseline)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
r2_base = r2_score(y_test, y_pred_baseline)

results = pd.DataFrame([
    {'Modelo': 'Naive Bayes (proxy regresion)', 'MAE': mae_nb, 'RMSE': rmse_nb, 'R2': r2_nb},
    {'Modelo': 'Baseline mediana train', 'MAE': mae_base, 'RMSE': rmse_base, 'R2': r2_base},
]).round(4)

print('Metricas del modelo:')
display(results)

Metricas del modelo:


,Modelo,MAE,RMSE,R2
0,Naive Bayes (proxy regresion),416.7355,896.8748,-0.0841
1,Baseline mediana train,230.1507,875.6788,-0.0334


### Analisis de resultados

Con los resultados obtenidos, el modelo **Naive Bayes (proxy de regresion)** presenta desempeno bajo para este problema: `MAE = 416.74`, `RMSE = 896.87` y `R2 = -0.0841`. Ademas, queda por debajo del baseline de referencia (`MAE = 230.15`, `RMSE = 875.68`, `R2 = -0.0334`).

Un `R2` negativo indica que el modelo explica menos variabilidad que una prediccion constante; por eso, en este experimento no logra capturar adecuadamente la estructura del precio. Esto es consistente con la naturaleza del metodo: se discretiza una variable continua, se asume independencia condicional entre predictores y luego se reconstruye el precio con medianas por bin, lo cual introduce perdida de informacion y errores grandes en valores intermedios/extremos.

**Conclusion de la actividad:** el enfoque es valido como implementacion academica de Naive Bayes para regresion aproximada y cumple la comparabilidad con LAB04 (mismo preprocesamiento y split), pero no es competitivo para prediccion de precio en este dataset. Para mejor desempeno, convienen modelos que capturen no linealidad e interacciones (por ejemplo, arboles, random forest o gradient boosting).

In [29]:
# Vista rapida de predicciones vs reales para inspeccion
comparison_preview = pd.DataFrame({
    'y_real': y_test.head(15).values,
    'y_pred_nb': y_pred_nb.head(15).values,
})
comparison_preview['error_abs'] = (comparison_preview['y_real'] - comparison_preview['y_pred_nb']).abs()

display(comparison_preview.round(2))

,y_real,y_pred_nb,error_abs
0,344.0,920.0,576.0
1,295.0,920.0,625.0
2,108.0,452.0,344.0
3,69.0,322.0,253.0
4,480.0,322.0,158.0
5,170.0,920.0,750.0
6,80.0,920.0,840.0
7,315.0,209.0,106.0
8,93.0,176.0,83.0
9,200.0,920.0,720.0
